# 04 — Play Success Model

**Trinity University Football Analytics**

This notebook defines play success using the EP/EPA framework and analyzes which situations, personnel groupings, and play types drive or hurt performance. No separate classifier is needed — a play is successful if it has **positive EPA** (outperformed the expected points for that situation).

**Input:** `all_df` with `ep` and `epa` from notebook 03

**Output:** Play-level success flags + coaching-facing summaries

---

### Why EPA as the Success Measure
A 3-yard gain on 3rd-and-2 is a success. The same gain on 3rd-and-10 is not. EPA captures this context automatically — positive EPA means the play outperformed the situation's expectation.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# all_df should be in memory from notebooks 01-03
# If running standalone, load from your exported Excel:
# all_df = pd.read_excel('/content/drive/MyDrive/TUFB_EPA/TUFB_EPA_Analysis.xlsx')
print(f"all_df shape: {all_df.shape}")
print(f"EPA null count: {all_df['epa'].isna().sum()}")

## 2. Define Play Success

A play is **successful** if EPA > 0 (outperformed expectation for that situation).

In [ ]:
# Filter to offense and defense only
od_df = all_df[all_df['odk'].isin(['o', 'd'])].copy()
od_df = od_df.dropna(subset=['epa']).reset_index(drop=True)

# Success flag: positive EPA = successful play
od_df['success'] = (od_df['epa'] > 0).astype(int)

print("Overall success rate by unit:")
print(od_df.groupby('odk')['success'].mean().round(3))

## 3. Success Rate by Down & Distance

In [ ]:
print("Offensive success rate by down:")
print(od_df[od_df['odk'] == 'o'].groupby('dn')['success'].mean().round(3))

print("\nOffensive EPA by down:")
print(od_df[od_df['odk'] == 'o'].groupby('dn')['epa'].mean().round(3))

## 4. Success Rate by Play Type

In [ ]:
play_type_summary = (
    od_df[od_df['odk'] == 'o']
    .groupby('play_type')
    .agg(
        plays=('epa', 'count'),
        mean_epa=('epa', 'mean'),
        success_rate=('success', 'mean')
    )
    .round(3)
    .sort_values('mean_epa', ascending=False)
)
print("Offensive play type summary:")
print(play_type_summary.to_string())

## 5. Success Rate by Personnel Grouping

In [ ]:
personnel_summary = (
    od_df[od_df['odk'] == 'o']
    .groupby('personnel')
    .agg(
        plays=('epa', 'count'),
        mean_epa=('epa', 'mean'),
        success_rate=('success', 'mean')
    )
    .round(3)
    .sort_values('plays', ascending=False)
)
print("Personnel grouping summary (Offense, min 10 plays):")
print(personnel_summary[personnel_summary['plays'] >= 10].to_string())

## 6. Season-Level EPA Trend

In [ ]:
# Extract season year from game_date
od_df['season'] = pd.to_datetime(od_df['game_date'], errors='coerce').dt.year

season_epa = (
    od_df.groupby(['season', 'odk'])['epa']
    .mean()
    .round(3)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
for unit, color, label in zip(['o', 'd'], ['steelblue', 'firebrick'], ['Offense', 'Defense']):
    unit_data = season_epa[season_epa['odk'] == unit]
    ax.plot(unit_data['season'], unit_data['epa'], marker='o', color=color, linewidth=2, label=label)

ax.axhline(0, color='black', linestyle='--', alpha=0.4)
ax.set_title('Mean EPA by Season', fontsize=14)
ax.set_xlabel('Season', fontsize=12)
ax.set_ylabel('Mean EPA', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. High-Impact Play Identification

Top positive and negative EPA plays — the plays that swung games the most.

In [ ]:
impact_cols = ['game_date', 'odk', 'dn', 'dist', 'play_type', 'result', 'gn_ls', 'ep', 'epa']

print("Top 10 highest EPA plays (Offense):")
print(
    od_df[od_df['odk'] == 'o']
    .nlargest(10, 'epa')[impact_cols]
    .to_string(index=False)
)

print("\nTop 10 lowest EPA plays (Offense — biggest negative impact):")
print(
    od_df[od_df['odk'] == 'o']
    .nsmallest(10, 'epa')[impact_cols]
    .to_string(index=False)
)

## 8. Success Rate Heatmap — Down × Distance Bucket

In [ ]:
off_df = od_df[od_df['odk'] == 'o'].copy()
off_df['dist'] = pd.to_numeric(off_df['dist'], errors='coerce')

dist_bins   = [0, 3, 7, 10, 15, 100]
dist_labels = ['0-3', '4-7', '8-10', '11-15', '15+']
off_df['dist_bucket'] = pd.cut(off_df['dist'], bins=dist_bins, labels=dist_labels)

heatmap_data = off_df.groupby(['dn', 'dist_bucket'])['success'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, ax=ax, linewidths=0.5
)
ax.set_title('Offensive Success Rate by Down & Distance', fontsize=14)
ax.set_xlabel('Distance to First Down', fontsize=12)
ax.set_ylabel('Down', fontsize=12)
plt.tight_layout()
plt.show()

## 9. Export Final Dataset

In [ ]:
# Merge success flag back to all_df
all_df = all_df.merge(
    od_df[['game_id', 'play', 'success']],
    on=['game_id', 'play'],
    how='left'
)

# Uncomment to export
# all_df.to_excel('/content/drive/MyDrive/TUFB_EPA/TUFB_EPA_Analysis.xlsx', sheet_name='PlayList', index=False)
# print('Exported successfully.')

print(f"Final dataset: {all_df.shape}")
print(all_df[['ep', 'epa', 'success']].describe().round(3))